# Dashboard data: one monthly table, the latest-month figures, and the period changes the browser must reproduce

The dashboard landing page filters, rebases and measures period changes in the browser. This notebook does all the calculation first, in Python, from the database,
and checks every latest-month figure against an independent source table. The browser result is later compared with the period-change table written here
(`tools/parity_test.py`, Playwright, tolerance 0.01).

**Read this first.** These are descriptions of past prices. Nothing here is a forecast. Only World Bank and FRED-derived series are used (no LME data).
Monthly prices are monthly averages. "Today's money" means US dollars with US CPI taken out, with the latest month as the base.
Run `python copper_database/build_db.py --raw .` first.

In [1]:
import sqlite3
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
con = sqlite3.connect("../copper_database/copper.db")
rd = lambda sql: pd.read_sql(sql, con)
written, checks = {}, []


def save(df, name):
    df.to_csv(f"../results/{name}.csv", index=False)
    df.to_sql(name, con, if_exists="replace", index=False)
    written[name] = (len(df), len(df.columns))


def check(fig_id, figure, expected, computed, tol=0.0, unit=""):
    """Record one figure; stop the notebook if it does not reproduce."""
    ok = (str(computed) == str(expected)) if isinstance(expected, str) else abs(float(computed) - float(expected)) <= tol
    checks.append({"figure_id": fig_id, "figure": figure, "expected_from_source_table": expected if isinstance(expected, str) else round(float(expected), 4),
                   "computed_here": computed if isinstance(computed, str) else round(float(computed), 4), "tolerance": tol, "unit": unit, "status": "PASS" if ok else "FAIL"})
    assert ok, f"{fig_id} does not reproduce: {figure}: source table {expected}, computed {computed}"

## 1. The monthly table
Prices come from the World Bank monthly file (staged table). Euro prices divide the dollar price by the FRED euro rate (euros start in 1999). The real series multiply by
the latest CPI over each month's CPI. October 2025 has no US CPI value, so the real values for that month stay empty and are never filled.

In [2]:
wb = rd("select substr(date,1,7) as month, commodity, value from stg_wb_prices_monthly where commodity in ('Copper','Aluminum','Gold','Tin','Crude oil, Brent')")
w = wb.pivot(index="month", columns="commodity", values="value").rename(columns={"Copper": "cu_usd", "Aluminum": "al_usd", "Gold": "gold_usd", "Tin": "tin_usd", "Crude oil, Brent": "brent_usd"})
fred = rd("select substr(date,1,7) as month, series_id, value from stg_fred_monthly where series_id in ('CPIAUCSL','EXUSEU')").pivot(index="month", columns="series_id", values="value")
panel = rd("select substr(month,1,7) as month, dollar_index_avg, copper_eur_t_avg from mart_monthly_panel").set_index("month")
roll = rd("select substr(month,1,7) as month, \"Broad dollar index\" as corr36 from res_dollar_rolling_correlation_36m").set_index("month")
ratio = rd("select substr(month,1,7) as month, cu_al_ratio as ratio from mart_cu_al_ratio").set_index("month")

d = w.join(fred).join(panel[["dollar_index_avg"]]).join(roll).join(ratio)
d = d.rename(columns={"dollar_index_avg": "dxy", "CPIAUCSL": "cpi", "EXUSEU": "usd_per_eur"}).sort_index()
d["cu_eur"] = d["cu_usd"] / d["usd_per_eur"]
latest = d.index[-1]
cpi_base = d.loc[latest, "cpi"]
for k in ("cu", "al", "gold", "tin", "brent"):
    d[f"{k}_real"] = d[f"{k}_usd"] * cpi_base / d["cpi"]
d["ratio_ma12"] = d["ratio"].rolling(12).mean()
for k in ("cu", "al", "gold", "tin", "brent"):
    d[f"{k}_rec_pct"] = 100 * d[f"{k}_real"] / d[f"{k}_real"].max()          # share of the metal's own highest real price since 1960
cols = ["cu_usd", "cu_eur", "cu_real", "al_usd", "al_real", "gold_usd", "gold_real", "tin_usd", "tin_real", "brent_usd", "brent_real", "dxy", "ratio", "ratio_ma12", "corr36", "cu_rec_pct", "al_rec_pct", "gold_rec_pct", "tin_rec_pct", "brent_rec_pct"]
out = d[cols].round(4).reset_index()
print(latest, len(out), "months;", out["month"].min(), "to", out["month"].max())
print("months with no real value:", out.loc[out["cu_real"].isna(), "month"].tolist())
save(out, "res_dash_series")

2026-08 800 months; 1960-01 to 2026-08
months with no real value: ['2025-10']


## 2. Latest-month figures (the KPI band), each checked against a source table that was not used to build them

In [3]:
case = rd("select fact_id, value from res_aluminium_case").set_index("fact_id")["value"]
rec = rd("select fact_id, month, value from res_story_copper_record_facts").set_index("fact_id")
lat = out.iloc[-1]
real_peak = out["cu_real"].max()
real_peak_month = out.loc[out["cu_real"].idxmax(), "month"]
share_real = 100 * lat["cu_real"] / real_peak
months_total = int(out["cu_usd"].notna().sum())
kpis = pd.DataFrame([
    ["latest_month", latest, "month", "last month of the World Bank copper series"],
    ["copper_usd_t", lat["cu_usd"], "USD per tonne", "World Bank monthly average as quoted"],
    ["copper_eur_t", lat["cu_eur"], "EUR per tonne", "dollar price divided by the FRED euro rate"],
    ["months_total", months_total, "months", "months in the copper series"],
    ["eur_months_total", int(out["cu_eur"].notna().sum()), "months", "months in the euro copper series (euros start in 1999)"],
    ["eur_is_record", float(lat["cu_eur"] >= out["cu_eur"].max()), "1 = yes", "latest month is the highest in euros"],
    ["copper_is_nominal_record", float(lat["cu_usd"] >= out["cu_usd"].max()), "1 = yes", "latest month is the highest as quoted"],
    ["real_share_of_record_pct", share_real, "percent", "latest real price over the highest real price"],
    ["real_peak_month", real_peak_month, "month", "month of the highest real price"],
    ["ratio_latest", lat["ratio"], "ratio", "copper price over aluminium price"],
    ["breakeven_ratio", float(case["breakeven_ratio"]), "ratio", "from NBS Circular 31 constants"],
    ["dxy_latest", lat["dxy"], "index", "broad dollar index, monthly average"],
    ["dxy_first_month", out.loc[out["dxy"].notna(), "month"].iloc[0], "month", "the dollar index starts here"],
    ["eur_first_month", out.loc[out["cu_eur"].notna(), "month"].iloc[0], "month", "euro prices start here"],
    ["cpi_base_month", latest, "month", "the real series use this month's prices"],
    ["copper_12m_change_pct", 100 * (lat["cu_usd"] / out["cu_usd"].iloc[-13] - 1), "percent", "latest month against the same month a year earlier, as quoted in dollars"],
    ["dxy_12m_change_pct", 100 * (lat["dxy"] / out["dxy"].iloc[-13] - 1), "percent", "dollar index, latest month against the same month a year earlier"],
    ["real_rank_latest", float((out["cu_real"] > lat["cu_real"]).sum() + 1), "rank", "place of the latest real price among all months with a real price (1 = highest)"],
    ["real_months_valid", float(out["cu_real"].notna().sum()), "months", "months with a real price (October 2025 has no US CPI)"],
], columns=["fact_id", "value", "unit", "note"])
save(kpis, "res_dash_kpis")
kp = kpis.set_index("fact_id")["value"]

# independent sources for each check
wb_cu = rd("select value from stg_wb_prices_monthly where commodity='Copper' and substr(date,1,7)=(select max(substr(date,1,7)) from stg_wb_prices_monthly where commodity='Copper')")["value"].iloc[0]
eur_panel = float(panel.loc[latest, "copper_eur_t_avg"])
check("D01", "copper price, latest month (USD per tonne)", wb_cu, float(kp["copper_usd_t"]), 0.0, "USD/t")
check("D02", "copper price, latest month (EUR per tonne), against the staged panel", eur_panel, float(kp["copper_eur_t"]), 0.5, "EUR/t")
check("D03", "months in the copper series", int(rec.loc["series_months", "value"]), months_total, 0, "months")
check("D15", "latest month is the highest in euros, against the euro result table of the story", float(rd("select value from res_euro_record where fact_id='eur_latest_is_record'")["value"].iloc[0]), float(kp["eur_is_record"]), 0, "1 = yes")
check("D16", "months in the euro series: January 1999 to the latest month, counted from the calendar", (int(latest[:4]) - 1999) * 12 + int(latest[5:7]), float(kp["eur_months_total"]), 0, "months")
check("D22", "place of the latest real price among all months (rank), against the story record facts (months above it, plus one)", float(rec.loc["real_months_above_latest", "value"]) + 1, float(kp["real_rank_latest"]), 0, "rank")
check("D23", "the one-year change of the dollar index, by hand from two months", 100 * (float(out["dxy"].iloc[-1]) / float(out["dxy"].iloc[-13]) - 1), float(kp["dxy_12m_change_pct"]), 0.001, "percent")
check("D24", "months with a real price are all months but one (October 2025)", float(months_total) - 1, float(kp["real_months_valid"]), 0, "months")
check("D04", "latest month is the highest as quoted", 1.0, float(kp["copper_is_nominal_record"]), 0, "1 = yes")
check("D05", "latest real price as a share of the real record (percent), against the story record facts",
      100 / (1 + float(rec.loc["real_peak_vs_latest_pct", "value"]) / 100), share_real, 0.05, "percent")
check("D06", "month of the real record, against the story record facts", str(rec.loc["real_peak_all", "month"])[:7], real_peak_month)
check("D07", "copper-to-aluminium ratio, latest month, against the result table of the aluminium chapter", float(case["ratio_latest"]), float(kp["ratio_latest"]), 0.0005, "ratio")
check("D08", "dollar index, latest month, against the staged panel", float(panel.loc[latest, "dollar_index_avg"]), float(kp["dxy_latest"]), 0.0001, "index")
check("D09", "the real series is the nominal series in latest-month money (latest real = latest nominal)", float(lat["cu_usd"]), float(lat["cu_real"]), 0.001, "USD/t")
check("D10", "US CPI has no value for October 2025: the real value stays empty", "True", str(bool(out.loc[out["month"] == "2025-10", "cu_real"].isna().iloc[0])))
cmp = pd.read_sql("select substr(month,1,7) as month, real_usd_t_aug2026 as r from res_story_copper_series", con).set_index("month")
dev = (out.set_index("month")["cu_real"] - cmp["r"]).abs().max()
check("D11", "real copper series against the story's real copper series (largest difference, USD per tonne; the story rounds to whole dollars)", 0.0, dev, 1.0, "USD/t")

## 3. Period changes: what the browser must reproduce
For each period (1 year, 5 years, 20 years, all) and each currency, the window starts N months before the latest month (all: the first month of the series).
Euro prices start in 1999, so a longer window starts there. Today's money is dollars only. The change is latest over start minus one, in percent. "Rebased" values set
the start month to 100. The dollar index starts in 2006, so its window starts there when the period reaches back further.

In [4]:
N = {"1y": 12, "5y": 60, "20y": 240, "all": None}
months = out["month"].tolist()
idx_latest = len(months) - 1
rows = []
for p, n in N.items():
    for cur in ("usd", "eur"):
        for prices in ("nominal", "real"):
            if cur == "eur" and prices == "real":
                continue                      # today's money is US inflation: dollars only
            col = {("usd", "nominal"): "cu_usd", ("eur", "nominal"): "cu_eur", ("usd", "real"): "cu_real"}[(cur, prices)]
            start_i = 0 if n is None else max(0, idx_latest - n)
            s = out[col]
            first_valid = s.first_valid_index()
            start_i = max(start_i, first_valid)
            # a real value can be missing in the start month (October 2025): move to the next month with a value
            while pd.isna(s.iloc[start_i]):
                start_i += 1
            row = {"period": p, "currency": cur, "prices": prices, "start_month": months[start_i], "copper_change_pct": round(100 * (s.iloc[-1] / s.iloc[start_i] - 1), 4)}
            if cur == "usd":
                for k in ("al", "gold", "tin", "brent"):
                    ser = out[f"{k}_usd" if prices == "nominal" else f"{k}_real"]
                    row[f"{k}_rebased_latest"] = round(100 * ser.iloc[-1] / ser.iloc[start_i], 4)
            # the dollar index and the ratio panels do not depend on the currency: they use the dollar window of the same period and prices
            su = out["cu_real" if prices == "real" else "cu_usd"]
            su_i = 0 if n is None else max(0, idx_latest - n)
            su_i = max(su_i, su.first_valid_index())
            while pd.isna(su.iloc[su_i]):
                su_i += 1
            dx = out["dxy"]
            ds = max(su_i, dx.first_valid_index())
            row["dxy_start_month"] = months[ds]
            row["dxy_change_pct"] = round(100 * (dx.iloc[-1] / dx.iloc[ds] - 1), 4)
            rs = out["ratio"]
            row["ratio_start"] = round(rs.iloc[su_i], 4)
            rows.append(row)
sc = rd("select substr(month,1,7) as month, dollar_index_pct_m as dx, copper_usd_pct_m as cu from mart_monthly_changes where dollar_index_pct_m is not null and copper_usd_pct_m is not null").set_index("month")
for r in rows:
    w = sc[sc.index > r["dxy_start_month"]]               # months after the window's first month (a change needs the month before)
    up, dn = w[w.dx > 0], w[w.dx < 0]
    r["dollar_rose_n"], r["copper_fell_when_rose_n"] = int(len(up)), int((up.cu < 0).sum())
    r["dollar_fell_n"], r["copper_rose_when_fell_n"] = int(len(dn)), int((dn.cu > 0).sum())
chg = pd.DataFrame(rows)
save(chg, "res_dash_changes")
print(chg.to_string())
chg_i = chg.set_index(["period", "currency", "prices"])
check("D12", "copper change over one year as quoted in USD, by hand from two months", 100 * (float(out["cu_usd"].iloc[-1]) / float(out["cu_usd"].iloc[-13]) - 1), float(chg_i.loc[("1y", "usd", "nominal"), "copper_change_pct"]), 0.001, "percent")
check("D13", "the one-year change in the opener of the story (result table of the uses chapter)", float(rd("select value from res_uses_facts where fact_id='copper_12m_change_pct'")["value"].iloc[0]),
      float(chg_i.loc[("1y", "usd", "nominal"), "copper_change_pct"]), 0.01, "percent")
check("D14", "euro window for 'all' starts in 1999", "1999-01", chg_i.loc[("all", "eur", "nominal"), "start_month"])

eu = rd("select sector, share_pct from copper_end_use order by share_pct desc")
eur = rd("select sector, share_pct from res_uses_end_use order by rank")
check("D17", "end-use shares add up to 100 (ICSG Factbook 2025, S31)", 100, float(eu["share_pct"].sum()), 0, "percent")
check("D18", "the six end-use shares in the result table equal the collected table, sector by sector (largest abs difference)", 0.0, float((eu.set_index("sector")["share_pct"] - eur.set_index("sector")["share_pct"]).abs().max()), 0, "percentage points")
cf = rd("select fact_id, value from copper_context_facts").set_index("fact_id")["value"]
check("D19", "wire 63, China 58 and recycled about one third come from the collected facts table with their source id", 63 + 58 + 33, float(cf["F06"] + cf["F01"] + cf["F02"]), 0, "sum of three percentages")
check("D20", "the 12-month average of the ratio in the latest month equals the mean of the last 12 ratios", float(out["ratio"].iloc[-12:].mean()), float(out["ratio_ma12"].iloc[-1]), 0.0002, "ratio")
check("D21", "each metal's share of its own real record is 100 in its record month and the five latest values match the interlude records table", 0.0,
      float(max(abs(out[f"{k}_rec_pct"].iloc[-1] - float(rd(f"select latest_pct_of_real_peak v from res_interlude_records where commodity='{n}'")["v"].iloc[0])) for k, n in (("cu", "copper"), ("al", "aluminium"), ("gold", "gold"), ("tin", "tin"), ("brent", "brent")))), 0.05, "percentage points")

   period currency   prices start_month  copper_change_pct  al_rebased_latest  gold_rebased_latest  tin_rebased_latest  brent_rebased_latest dxy_start_month  dxy_change_pct  ratio_start  dollar_rose_n  copper_fell_when_rose_n  dollar_fell_n  copper_rose_when_fell_n
0      1y      usd  nominal     2025-08            48.1489           125.1829             130.9679            163.6963              133.2845         2025-08         -1.4303       3.7235              5                        2              7                        6
1      1y      usd     real     2025-08            43.3426           121.1217             126.7190            158.3856              128.9603         2025-08         -1.4303       3.7235              5                        2              7                        6
2      1y      eur  nominal     2025-08            48.8262                NaN                  NaN                 NaN                   NaN         2025-08         -1.4303       3.7235              5  

## 4. Write the check table

In [5]:
fc = pd.DataFrame(checks)
save(fc, "res_dash_figure_checks")
print(fc[["figure_id", "figure", "status"]].to_string())
print({k: v for k, v in written.items()})

   figure_id                                                                                                                             figure status
0        D01                                                                                         copper price, latest month (USD per tonne)   PASS
1        D02                                                               copper price, latest month (EUR per tonne), against the staged panel   PASS
2        D03                                                                                                        months in the copper series   PASS
3        D15                                                   latest month is the highest in euros, against the euro result table of the story   PASS
4        D16                                             months in the euro series: January 1999 to the latest month, counted from the calendar   PASS
5        D22                 place of the latest real price among all months (rank), against t


{'res_dash_series': (800, 21), 'res_dash_kpis': (19, 4), 'res_dash_changes': (12, 16), 'res_dash_figure_checks': (24, 7)}
